# 13 — Streaming Ingestion

## 1. Create the Streaming Storage Layer

We create a dedicated schema for streaming data.

A Unity Catalog Volume will store the files that arrive over time before they are ingested with Auto Loader.

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.streaming_data
""")

spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.streaming_data.streaming_files
""")

print("Streaming schema created")
print("Streaming Volume created")

Streaming schema created
Streaming Volume created


## 2. Create the Streaming Folders

We create three folders:

- `incoming_orders`: new transaction files arrive here
- `schema`: Auto Loader stores the detected schema here
- `checkpoints`: Spark remembers which files were already processed

In [0]:
base_path = (
    "/Volumes/workspace/streaming_data/"
    "streaming_files"
)

incoming_path = f"{base_path}/incoming_orders"
schema_path = f"{base_path}/schema/orders"
checkpoint_path = f"{base_path}/checkpoints/orders"

dbutils.fs.mkdirs(incoming_path)
dbutils.fs.mkdirs(schema_path)
dbutils.fs.mkdirs(checkpoint_path)

print("Incoming folder:", incoming_path)
print("Schema folder:", schema_path)
print("Checkpoint folder:", checkpoint_path)

Incoming folder: /Volumes/workspace/streaming_data/streaming_files/incoming_orders
Schema folder: /Volumes/workspace/streaming_data/streaming_files/schema/orders
Checkpoint folder: /Volumes/workspace/streaming_data/streaming_files/checkpoints/orders


## 3. Simulate New Incoming Orders

The original Instacart dataset is historical, so it does not receive real-time transactions.

To demonstrate streaming ingestion, we simulate new order files arriving over time.

Each new file represents a small batch of new customer orders.

In [0]:
from pyspark.sql import Row

new_orders_batch_1 = [
    Row(
        order_id=4000001,
        user_id=101,
        eval_set="stream",
        order_number=21,
        order_dow=2,
        order_hour_of_day=10,
        days_since_prior_order=7.0
    ),
    Row(
        order_id=4000002,
        user_id=202,
        eval_set="stream",
        order_number=15,
        order_dow=4,
        order_hour_of_day=17,
        days_since_prior_order=10.0
    ),
    Row(
        order_id=4000003,
        user_id=303,
        eval_set="stream",
        order_number=8,
        order_dow=6,
        order_hour_of_day=13,
        days_since_prior_order=5.0
    )
]

batch_1_df = spark.createDataFrame(
    new_orders_batch_1
)

batch_1_path = (
    f"{incoming_path}/batch_1"
)

(
    batch_1_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(batch_1_path)
)

print("First incoming order batch created")
print("Location:", batch_1_path)

First incoming order batch created
Location: /Volumes/workspace/streaming_data/streaming_files/incoming_orders/batch_1


## 4. Ingest New Orders with Auto Loader

Auto Loader watches the incoming folder for new files.

It only processes files that have not already been ingested.

The data is written to a Delta table in Unity Catalog:

`workspace.streaming_data.order_events`

In [0]:
from pyspark.sql import functions as F

streaming_orders_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .load(incoming_path)
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_ingested_at",
        F.current_timestamp()
    )
)

query = (
    streaming_orders_df
    .writeStream
    .option(
        "checkpointLocation",
        checkpoint_path
    )
    .trigger(availableNow=True)
    .toTable(
        "workspace.streaming_data.order_events"
    )
)

query.awaitTermination()

print("Streaming ingestion completed")

Streaming ingestion completed


## 5. Validate the Streaming Table

We check that the first incoming file was loaded correctly into the Delta table.

The checkpoint will allow Auto Loader to remember this file and avoid processing it again.

In [0]:
streaming_table = (
    spark.table(
        "workspace.streaming_data.order_events"
    )
)

print(
    "Rows in streaming table:",
    streaming_table.count()
)

display(streaming_table)

Rows in streaming table: 6


order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,_rescued_data,_source_file,_ingested_at
4000001,101,stream,21,2,10,7.0,null,/Volumes/workspace/streaming_data/streaming_files/incoming_orders/batch_1/part-00000-tid-3815194212045783136-f56afe4e-1c01-4221-9e9f-285f315f2530-131-1-c000.csv,2026-08-19T09:34:31.644Z
4000002,202,stream,15,4,17,10.0,null,/Volumes/workspace/streaming_data/streaming_files/incoming_orders/batch_1/part-00000-tid-3815194212045783136-f56afe4e-1c01-4221-9e9f-285f315f2530-131-1-c000.csv,2026-08-19T09:34:31.644Z
4000003,303,stream,8,6,13,5.0,null,/Volumes/workspace/streaming_data/streaming_files/incoming_orders/batch_1/part-00000-tid-3815194212045783136-f56afe4e-1c01-4221-9e9f-285f315f2530-131-1-c000.csv,2026-08-19T09:34:31.644Z
4000004,404,stream,12,1,9,6.0,null,/Volumes/workspace/streaming_data/streaming_files/incoming_orders/batch_2/part-00000-tid-4560152753979060766-b79454c5-2c45-4b9c-8a80-c9c3d6080206-136-1-c000.csv,2026-08-19T09:38:31.233Z
4000005,505,stream,18,3,14,8.0,null,/Volumes/workspace/streaming_data/streaming_files/incoming_orders/batch_2/part-00000-tid-4560152753979060766-b79454c5-2c45-4b9c-8a80-c9c3d6080206-136-1-c000.csv,2026-08-19T09:38:31.233Z
4000006,606,stream,10,5,19,4.0,null,/Volumes/workspace/streaming_data/streaming_files/incoming_orders/batch_2/part-00000-tid-4560152753979060766-b79454c5-2c45-4b9c-8a80-c9c3d6080206-136-1-c000.csv,2026-08-19T09:38:31.233Z


## 6. Test Incremental Streaming

We add a second file containing new orders.

When Auto Loader runs again, the checkpoint ensures that only the new file is processed and the first file is not loaded again.

In [0]:
new_orders_batch_2 = [
    Row(
        order_id=4000004,
        user_id=404,
        eval_set="stream",
        order_number=12,
        order_dow=1,
        order_hour_of_day=9,
        days_since_prior_order=6.0
    ),
    Row(
        order_id=4000005,
        user_id=505,
        eval_set="stream",
        order_number=18,
        order_dow=3,
        order_hour_of_day=14,
        days_since_prior_order=8.0
    ),
    Row(
        order_id=4000006,
        user_id=606,
        eval_set="stream",
        order_number=10,
        order_dow=5,
        order_hour_of_day=19,
        days_since_prior_order=4.0
    )
]

batch_2_df = spark.createDataFrame(new_orders_batch_2)

batch_2_path = f"{incoming_path}/batch_2"

(
    batch_2_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(batch_2_path)
)

print("Second incoming batch created")

Second incoming batch created


In [0]:
streaming_table = spark.table(
    "workspace.streaming_data.order_events"
)

print("Rows after batch 2:", streaming_table.count())

display(
    streaming_table
    .orderBy("order_id")
)

Rows after batch 2: 3


order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,_rescued_data,_source_file,_ingested_at
4000001,101,stream,21,2,10,7.0,null,/Volumes/workspace/streaming_data/streaming_files/incoming_orders/batch_1/part-00000-tid-3815194212045783136-f56afe4e-1c01-4221-9e9f-285f315f2530-131-1-c000.csv,2026-08-19T09:34:31.644Z
4000002,202,stream,15,4,17,10.0,null,/Volumes/workspace/streaming_data/streaming_files/incoming_orders/batch_1/part-00000-tid-3815194212045783136-f56afe4e-1c01-4221-9e9f-285f315f2530-131-1-c000.csv,2026-08-19T09:34:31.644Z
4000003,303,stream,8,6,13,5.0,null,/Volumes/workspace/streaming_data/streaming_files/incoming_orders/batch_1/part-00000-tid-3815194212045783136-f56afe4e-1c01-4221-9e9f-285f315f2530-131-1-c000.csv,2026-08-19T09:34:31.644Z


## 7. Conclusion

Auto Loader successfully ingested new order files incrementally using Spark Structured Streaming.

The first batch added 3 orders.

After a second file arrived, Auto Loader processed only the new file and the Delta table increased from 3 to 6 rows without duplicating the first batch.

The streaming data is stored in:

`workspace.streaming_data.order_events`

This demonstrates incremental ingestion using Auto Loader, Structured Streaming, Delta Lake, Unity Catalog, and checkpointing.